# Stock Market Predictor & Financial Intelligence Agent

### A Guided Walkthrough of the Full System, End to End

This notebook walks through all five phases of the project in a single place: a
trained NLP classifier, a semantic news search index, a machine-learning price
forecast, an AI agent that combines both into a written brief, and the live
deployed API that serves it all. Every code cell below calls the **real**
project modules — `nlp_router.py`, `vector_store.py`, `quant_baseline.py`,
`chart_utils.py`, and `langgraph_agent.py` — the exact same code that runs in
production, not a simplified reimplementation. Markdown cells alongside each
step explain *what* is happening and *why*, aimed at a reader who may not
have a deep machine learning background.

**Before running:**

1. This notebook must sit in the same folder as `main.py`, `nlp_router.py`,
   `vector_store.py`, `quant_baseline.py`, `chart_utils.py`, and
   `langgraph_agent.py` — it imports them directly.
2. Install the notebook-only extras (not part of the production API's own
   dependency list): `pip install python-dotenv requests jupyter ipykernel`
   — everything else comes from `requirements.txt`.
3. A `.env` file with a valid `OPENAI_API_KEY` unlocks the full experience.
   Without one, Phases 2, 4, and 5 below still run, but in a clearly-labeled
   demo mode instead of making real OpenAI API calls — nothing will crash.
4. A trained classifier in `artifacts/` (see `train_classifier.py`) unlocks
   real predictions in Phase 1. Without one, Phase 1 runs in the project's
   built-in **Mock Mode** and says so explicitly, rather than pretending.


## Background

Two familiar approaches exist for reasoning about markets. **Quantitative
analysis** looks at historical numbers — prices, volumes, volatility — and
fits a model to extrapolate a short-term trend; it is objective and
reproducible, but blind to real-world events like an earnings surprise or a
regulatory action. **Qualitative analysis** looks at news and commentary,
capturing that real-world context, but is hard to turn into a specific,
searchable, quantifiable signal on its own.

This project combines both: a numerical anchor from a quantitative model
(Phase 3), and real-world grounding from a topic-classified, semantically
searchable news corpus (Phases 1 and 2), reconciled by a language model
(Phase 4) that explains *why* the two signals agree or disagree, rather than
presenting either one in isolation.

> **This is an educational and portfolio engineering project, not a
> financial product.** Short-horizon stock price movements are famously
> difficult to predict from historical price data alone — that is a
> well-established property of markets, not a flaw specific to this
> project's model choice. Nothing produced by this notebook or the live
> system should be used as the sole basis for a real financial decision.


## The Five Phases, at a Glance

| Phase | Module | What it does |
|---|---|---|
| 1 | `nlp_router.py` | Classifies a raw headline into one of 20 financial topics (Earnings, M&A, Macro, ...) |
| 2 | `vector_store.py` | Indexes topic-tagged news into a searchable vector database, filterable by topic |
| 3 | `quant_baseline.py` | Trains a fresh XGBoost model on recent price history to forecast tomorrow's close |
| 4 | `langgraph_agent.py` | A tool-calling AI agent that combines Phases 2 and 3 into one written brief |
| 5 | `main.py` | The FastAPI service that wraps everything behind a live HTTP API |

Data flows roughly like this: raw headlines get tagged by Phase 1, tagged
headlines get indexed by Phase 2, and at request time Phase 4's agent calls
Phase 3 for a number and Phase 2 for context, then writes the final answer.
Phase 5 is the container that keeps this running continuously as a web
service. The sections below walk through each phase in that same order,
with real, runnable code.


## A Few Terms, Defined

A handful of concepts come up repeatedly below — useful to have a working
definition of each before diving in:

- **Bag-of-Words** — representing text as a vector of word counts, ignoring
  grammar and order. Simple, fast, and what Phase 1's classifier is built on.
- **Embedding** — a numeric vector representation of text such that similar
  meanings produce similar vectors, which is what makes semantic search
  possible in Phase 2.
- **RAG (Retrieval-Augmented Generation)** — giving a language model access
  to retrieved, relevant documents at answer time, so its response is
  grounded in current source material instead of relying purely on what it
  memorized during training.
- **Regression** — predicting a continuous number (a price), as opposed to
  classification, which predicts a category. Phase 3 is a regression task;
  Phase 1 is a classification task.
- **Agent** — a language model run in a loop with access to tools (Python
  functions it can choose to call), deciding for itself when to call which
  tool before producing a final answer. That is what Phase 4 is.


## Setup

Load environment variables, quietly suppress noisy library warnings so the
output below stays readable, and check whether an OpenAI key is configured
— every later phase checks the `HAS_OPENAI_KEY` flag set here to decide
whether to make a real API call or explain what it would have done.


In [ ]:
import os
import sys
import json
import warnings

warnings.filterwarnings("ignore")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")

sys.path.insert(0, os.getcwd())

from dotenv import load_dotenv
load_dotenv()

HAS_OPENAI_KEY = bool(os.getenv("OPENAI_API_KEY"))
DEMO_TICKER = "AAPL"

print(f"OPENAI_API_KEY configured: {HAS_OPENAI_KEY}")
if not HAS_OPENAI_KEY:
    print("No key found - Phases 2, 4, and 5 below will run in a clearly-labeled")
    print("demo mode instead of making real OpenAI API calls. Add one to a .env")
    print("file in this folder (OPENAI_API_KEY=...) to unlock the full run.")


---
## Phase 1 — NLP Topic Classifier

Before any headline can be indexed into a searchable knowledge base, it
needs a topic label — "Earnings," "Fed / Central Banks," "M&A," and so on —
so that later retrieval can be scoped to a relevant category instead of
searching blindly across everything. That label is what this phase produces.

**The approach: Bag-of-Words + a small neural network.** Text is first
converted into a fixed-length vector by counting how often each word in a
15,000-word vocabulary appears (ignoring word order entirely), using
scikit-learn's `CountVectorizer`. That vector is then fed through a small
Keras feedforward network — `Dense(256, relu)` → `Dropout(0.3)` →
`Dense(128, relu)` → `Dropout(0.3)` → `Dense(20, softmax)` — which learns to
map word-count patterns to one of 20 topic categories.

**The dataset:** [`zeroshot/twitter-financial-news-topic`](https://huggingface.co/datasets/zeroshot/twitter-financial-news-topic) on
Hugging Face — 21,107 financial tweets across 20 categories (Analyst
Update, Fed / Central Banks, Earnings, M&A, Macro, and so on). The model
shipped with this project was trained on this real dataset, scoring a
**Weighted F1 of 0.8118** on the held-out validation split — a real,
honestly-measured number, not an assumption. Strongest categories include
Dividend (0.96 F1) and Earnings (0.94 F1); the weakest, Gold / Metals /
Materials (0.53 F1), simply has the fewest labeled training examples.

The cell below imports the real `nlp_router` module and calls its
`warm_up()` function — the same call `main.py` makes at API startup. If no
trained model is found in `artifacts/`, the module's own built-in **Mock
Mode** kicks in automatically: it logs a clear warning and returns
random placeholder topics instead of crashing, which is exactly what keeps
the rest of the pipeline runnable even before training is finished.


In [ ]:
import nlp_router

nlp_router.warm_up()

if nlp_router.MOCK_MODE:
    print()
    print("Running in MOCK MODE: no trained classifier found in artifacts/,")
    print("so the topic predictions below are randomly chosen placeholders,")
    print("not real model output. Run train_classifier.py (see the README's")
    print("'Training the NLP Classifier' section) to get real predictions here.")
else:
    print()
    print("Real trained classifier loaded - predictions below are genuine model output.")


### The Preprocessing Pipeline, Step by Step

Before text reaches the vectorizer, it passes through the same cleanup and
tokenization functions used during training (`train_classifier.py` imports
these exact functions from `nlp_router.py` for that reason — reusing one
pipeline for both training and serving eliminates a common, hard-to-detect
bug class called *train/serve skew*). Here is what each stage actually does
to one example headline:


In [ ]:
sample_text = "$AAPL smashes Q3 earnings estimates, guidance raised for FY24! https://example.com/news #Earnings"

cleaned = nlp_router.clean_text(sample_text)
tokens = nlp_router.tokenize_and_lemmatize(cleaned)
processed = nlp_router.preprocess(sample_text)

print("Raw text:        ", sample_text)
print("After clean_text:", cleaned)
print("After tokenizing/lemmatizing:", tokens)
print("Final (joined for the vectorizer):", processed)


### Classifying Real Example Headlines

Now the full pipeline, end to end, on a handful of varied examples covering
different topics:


In [ ]:
sample_headlines = [
    "$AAPL smashes Q3 earnings estimates, guidance raised for FY24",
    "Fed signals possible rate cuts amid cooling inflation data",
    "Breaking: $TSLA recalls 200k vehicles over autopilot software bug",
    "Chevron to acquire Hess Corporation in $53 billion all-stock merger deal",
    "Company announces quarterly dividend increase of 8 percent",
]

print(f"{'Predicted Topic':<28} {'Confidence':>10}   Headline")
print("-" * 100)
for headline in sample_headlines:
    tagged = nlp_router.tag_financial_news_detailed(headline)
    print(f"{tagged.topic:<28} {tagged.confidence:>9.1%}   {headline}")


---
## Phase 2 — Vector Database & Retrieval-Augmented Generation (RAG)

A language model on its own only knows what it learned during training and
has no access to live, current information. **RAG** solves this by
retrieving relevant documents from a database and handing them to the model
at answer time, so its response is grounded in real, current material.

To find text that is *relevant* rather than just *lexically identical*,
each document is converted into an **embedding** — a vector produced by a
model trained so that similar meanings produce similar vectors (this
project uses OpenAI's `text-embedding-3-small`). Documents and their
embeddings are stored in **ChromaDB**, running here in local, in-memory
mode. Critically, each document's Phase 1 topic label is stored as
**metadata** alongside its embedding, and search can be constrained to an
exact topic *before* ranking by similarity — which is what stops a query
about "merger" sentiment from accidentally pulling back an unrelated
"earnings" story that just happens to share similar financial vocabulary.

The cell below builds a small in-memory store from the project's bundled
sample corpus and runs three searches: unfiltered, filtered to a matching
topic, and filtered to a topic that should return nothing at all — proving
the metadata filter is genuinely enforced, not just semantic similarity
doing the work by coincidence.


In [ ]:
import vector_store

if not HAS_OPENAI_KEY:
    print("Skipping the live Phase 2 demo - OPENAI_API_KEY not configured")
    print("(OpenAIEmbeddings requires it). Here is the sample corpus this")
    print("phase would index instead:")
    for item in vector_store.MOCK_NEWS_ITEMS[:3]:
        print(" -", item)
else:
    try:
        store = vector_store.build_vector_store(vector_store.MOCK_NEWS_ITEMS)

        print("Unfiltered search: 'company financial results'")
        for r in vector_store.search_financial_news("company financial results"):
            print(" ", r)

        print()
        print("Filtered search (topic='Earnings'): 'revenue growth'")
        for r in vector_store.search_financial_news("revenue growth", topic="Earnings"):
            print(" ", r)

        print()
        print("Filtered search that should return nothing (topic='Crypto'): 'bitcoin price surge'")
        empty = vector_store.search_financial_news("bitcoin price surge", topic="Crypto")
        print(f"  Result count: {len(empty)} (expected 0 - the metadata filter is genuinely enforced)")
    except Exception as exc:
        print(f"Phase 2 live call failed ({exc}); check OPENAI_API_KEY and your network connection.")


---
## Phase 3 — Quantitative Baseline (XGBoost)

This phase produces the numerical half of the final brief: a next-trading-day
closing price forecast, trained fresh from the ticker's own recent price
history at request time.

**XGBoost** ("Extreme Gradient Boosting") builds an ensemble of decision
trees sequentially, where each new tree is trained to correct the errors of
the trees before it — a strong, fast-training default for small tabular
datasets. `quant_baseline.py` downloads six months of daily price data and
engineers four features from the raw closing price: `Close` itself, a
7-day and 14-day moving average (`MA_7`, `MA_14`), and 14-day rolling
volatility. The target is `Close` shifted forward by one trading day.

The most recent trading day is deliberately kept in the dataset with its
target left unknown (tomorrow hasn't happened yet) — every other row, which
has a real historical outcome, trains the model, and the trained model then
runs once on today's real features to forecast tomorrow's close. This is a
genuinely forward-looking forecast, not a backtest.

The cell below calls the real forecasting function, then hands its output
straight to `chart_utils.py` — the same module that powers the live API's
`GET /chart/{ticker}` endpoint — to render the price history with the
forecast plotted as a distinct point, displayed right here in the notebook.


In [ ]:
import quant_baseline
import chart_utils
import base64
from IPython.display import Image, display

try:
    forecast, price_history = quant_baseline.predict_stock_baseline_with_history(DEMO_TICKER)

    print(f"Last 5 trading days of {DEMO_TICKER} closing price:")
    print(price_history.tail(5).to_string())
    print()
    print("Forecast:")
    print(json.dumps(forecast, indent=2))

    chart_data_uri = chart_utils.generate_forecast_chart_base64(
        ticker=DEMO_TICKER,
        price_history=price_history,
        predicted_price=forecast["predicted_price"],
    )
    _header, encoded = chart_data_uri.split(",", 1)
    display(Image(data=base64.b64decode(encoded)))
except Exception as exc:
    print(f"Could not fetch live price data for {DEMO_TICKER} ({exc}).")
    print("This cell needs a working internet connection to Yahoo Finance to run live.")


---
## Phase 4 — Agentic Orchestration (LangGraph)

An **agent**, here, is a language model given a defined set of **tools** —
Python functions it can choose to call — and run in a loop: it reads the
conversation so far, decides whether it needs more information, calls a
tool if so, reads the result, and repeats until it has enough to answer.
That is different from a plain chatbot call, which only ever produces text
and cannot fetch new information mid-response.

This agent has exactly two tools: `get_quantitative_forecast` (wraps Phase
3) and `search_financial_news` (wraps Phase 2). Its control flow is an
explicit state graph built with **LangGraph** — an `agent` node (the LLM,
`gpt-4o-mini`) and a `tools` node, connected so the LLM can call a tool,
read the result, and decide again whether it needs another tool or is
ready to answer. Its system prompt instructs it to always call the
forecast tool first, the news tool second, and to synthesize both into a
brief that explicitly states whether the news corroborates or conflicts
with the number — never issuing a definitive buy/sell instruction.

The cell below runs the real agent end to end on a natural-language
question, exactly as the live `/predict` endpoint does internally.


In [ ]:
import langgraph_agent

if not HAS_OPENAI_KEY:
    print("Skipping the live Phase 4 demo - OPENAI_API_KEY not configured.")
else:
    try:
        brief = langgraph_agent.run_financial_agent(
            f"What is your outlook for {DEMO_TICKER} for the next trading day?"
        )
        print(brief)
    except Exception as exc:
        print(f"Phase 4 live call failed ({exc}).")


---
## Phase 5 — API Layer & Deployment

Everything above is wrapped by `main.py` into a small FastAPI application
with three endpoints, then packaged into a Docker image and deployed on a
live cloud instance:

| Endpoint | Method | Purpose |
|---|---|---|
| `/health` | GET | Liveness/readiness probe |
| `/predict` | POST | Runs the full Phase 1-4 pipeline, returns `{ticker, query, financial_brief, chart_image}` |
| `/chart/{ticker}` | GET | Returns the Phase 3 chart as a raw, browser-viewable PNG |

A startup routine warms up every heavy component (the classifier, the
vector store, the agent's LLM client) once when the container starts, so
the first real request doesn't pay model-load latency, and a background
job re-fetches and re-tags live headlines every hour so Phase 2's corpus
stays current. This phase is a long-running server process, so it isn't
run inside this notebook — instead, the cell below tries calling the
already-deployed live instance directly, the same way any external client
would.


In [ ]:
import requests

LIVE_API_BASE = "http://3.80.124.207:8000"

try:
    health = requests.get(f"{LIVE_API_BASE}/health", timeout=5).json()
    print("Live deployment health check:", health)

    resp = requests.post(
        f"{LIVE_API_BASE}/predict",
        json={
            "ticker": DEMO_TICKER,
            "query": "What is your outlook for the next trading day?",
        },
        timeout=90,
    ).json()
    print()
    print("Live financial_brief:")
    print(resp.get("financial_brief"))
except Exception as exc:
    print(f"Could not reach the live deployment at {LIVE_API_BASE} ({exc}).")
    print("This is expected if the EC2 instance is currently stopped, or if")
    print("this notebook is running somewhere without outbound internet access.")


---
## Summary

This notebook exercised all five phases of the real, deployed system in one
place: a trained topic classifier (Phase 1), a metadata-filtered semantic
search index (Phase 2), a fresh-trained price forecast with its own chart
(Phase 3), an AI agent that combines both into a written brief (Phase 4),
and — if reachable — the actual live API serving all of it (Phase 5).

For the full technical write-up — architecture diagrams, every algorithm
explained in more depth, the complete training and deployment process, and
an honest discussion of this system's real limitations — see the project's
`README.md`. As a reminder: this is a portfolio engineering project, not a
financial product, and nothing it produces should be the sole basis for a
real financial decision.
